<a href="https://colab.research.google.com/github/Hiroj12b/Cn6005/blob/main/apriori_week9_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
from itertools import combinations

# -----------------------------
# 1. Dataset (Week 9 example)
# -----------------------------
transactions = [
    {"Mango", "Onion", "Nintendo", "Key-chain", "Eggs", "Yo-yo"},          # T1
    {"Doll", "Onion", "Nintendo", "Key-chain", "Eggs", "Yo-yo"},          # T2
    {"Mango", "Apple", "Key-chain", "Eggs"},                              # T3
    {"Mango", "Umbrella", "Corn", "Key-chain", "Yo-yo"},                  # T4
    {"Corn", "Onion", "Key-chain", "Ice-cream", "Eggs"},                  # T5
]

min_support = 0.60   # 60%
min_confidence = 0.80  # 80%


# -----------------------------------
# 2. Helper: support of an itemset
# -----------------------------------
def support(transactions, itemset):
    """
    Compute support of an itemset as a fraction of total transactions.
    itemset: a frozenset of items
    """
    count = 0
    for t in transactions:
        if itemset.issubset(t):
            count += 1
    return count / len(transactions)


# -----------------------------------
# 3. Generate frequent 1-itemsets
# -----------------------------------
def get_frequent_1_itemsets(transactions, min_support):
    item_counts = {}
    for t in transactions:
        for item in t:
            itemset = frozenset([item])
            item_counts[itemset] = item_counts.get(itemset, 0) + 1

    freq_1 = {}
    total = len(transactions)
    for itemset, count in item_counts.items():
        s = count / total
        if s >= min_support:
            freq_1[itemset] = s
    return freq_1


# -----------------------------------
# 4. Candidate generation (join step)
# -----------------------------------
def apriori_gen(prev_frequent_itemsets, k):
    """
    Generate candidate k-itemsets from (k-1)-itemsets.
    prev_frequent_itemsets: list of frozenset
    """
    candidates = set()
    prev_list = list(prev_frequent_itemsets)

    for i in range(len(prev_list)):
        for j in range(i + 1, len(prev_list)):
            # join step: union two (k-1)-itemsets
            union_set = prev_list[i] | prev_list[j]
            if len(union_set) == k:
                candidates.add(union_set)

    return candidates


# -----------------------------------
# 5. Full Apriori algorithm
# -----------------------------------
def apriori(transactions, min_support):
    """
    Returns:
        all_frequent_itemsets: dict {frozenset: support}
    """
    all_frequent_itemsets = {}

    # Frequent 1-itemsets
    L1 = get_frequent_1_itemsets(transactions, min_support)
    current_L = L1
    all_frequent_itemsets.update(L1)
    k = 2

    # Generate Lk from L(k-1)
    while current_L:
        candidates_k = apriori_gen(current_L.keys(), k)
        freq_k = {}

        for c in candidates_k:
            s = support(transactions, c)
            if s >= min_support:
                freq_k[c] = s

        if not freq_k:
            break

        all_frequent_itemsets.update(freq_k)
        current_L = freq_k
        k += 1

    return all_frequent_itemsets


# -----------------------------------
# 6. Generate association rules
# -----------------------------------
def generate_rules(frequent_itemsets, min_confidence):
    """
    frequent_itemsets: dict {frozenset: support}
    Returns:
        list of (antecedent, consequent, support, confidence, lift)
    """
    rules = []
    itemsets = [fs for fs in frequent_itemsets.keys() if len(fs) >= 2]

    for itemset in itemsets:
        itemset_support = frequent_itemsets[itemset]

        # all non-empty proper subsets as antecedents
        for r in range(1, len(itemset)):
            for antecedent in combinations(itemset, r):
                antecedent = frozenset(antecedent)
                consequent = itemset - antecedent

                antecedent_support = frequent_itemsets.get(antecedent)
                consequent_support = frequent_itemsets.get(consequent)

                # skip if we don't have support stored (shouldn't happen if apriori was complete)
                if antecedent_support is None or consequent_support is None:
                    continue

                confidence = itemset_support / antecedent_support
                if confidence >= min_confidence:
                    lift = confidence / consequent_support
                    rules.append((antecedent, consequent,
                                  itemset_support, confidence, lift))
    return rules


# -----------------------------------
# 7. Run everything & print results
# -----------------------------------
if __name__ == "__main__":
    # Step 1: find frequent itemsets
    frequent_itemsets = apriori(transactions, min_support)

    print("=== FREQUENT ITEMSETS (support ≥", min_support, ") ===")
    for itemset, s in sorted(frequent_itemsets.items(), key=lambda x: (len(x[0]), x[0])):
        print(f"{set(itemset)}  --> support = {s:.2f}")

    # Step 2: generate strong rules
    rules = generate_rules(frequent_itemsets, min_confidence)

    print("\n=== STRONG ASSOCIATION RULES (confidence ≥", min_confidence, ") ===")
    for ant, cons, supp, conf, lift in rules:
        print(f"{set(ant)}  ->  {set(cons)}")
        print(f"   support = {supp:.2f}, confidence = {conf:.2f}, lift = {lift:.2f}")


=== FREQUENT ITEMSETS (support ≥ 0.6 ) ===
{'Yo-yo'}  --> support = 0.60
{'Eggs'}  --> support = 0.80
{'Mango'}  --> support = 0.60
{'Onion'}  --> support = 0.60
{'Key-chain'}  --> support = 1.00
{'Key-chain', 'Mango'}  --> support = 0.60
{'Key-chain', 'Eggs'}  --> support = 0.80
{'Onion', 'Key-chain'}  --> support = 0.60
{'Yo-yo', 'Key-chain'}  --> support = 0.60
{'Onion', 'Eggs'}  --> support = 0.60
{'Onion', 'Key-chain', 'Eggs'}  --> support = 0.60

=== STRONG ASSOCIATION RULES (confidence ≥ 0.8 ) ===
{'Mango'}  ->  {'Key-chain'}
   support = 0.60, confidence = 1.00, lift = 1.00
{'Key-chain'}  ->  {'Eggs'}
   support = 0.80, confidence = 0.80, lift = 1.00
{'Eggs'}  ->  {'Key-chain'}
   support = 0.80, confidence = 1.00, lift = 1.00
{'Onion'}  ->  {'Key-chain'}
   support = 0.60, confidence = 1.00, lift = 1.00
{'Yo-yo'}  ->  {'Key-chain'}
   support = 0.60, confidence = 1.00, lift = 1.00
{'Onion'}  ->  {'Eggs'}
   support = 0.60, confidence = 1.00, lift = 1.25
{'Onion'}  ->  {'Key-ch